# MELD preparation (run once on Kaggle)
Downloads MELD (~10 GB), extracts audio to 16 kHz mono wav (~1.5 GB),
builds the metadata csv, and leaves wavs + metadata in /kaggle/working.
Afterwards: Save Version, then Output tab -> New Dataset -> name it 'meld-wavs'.
No GPU needed for this notebook, CPU is fine.

In [ ]:
# 1. get the project code
!git clone https://github.com/Prathikesh/cognivoice-component-d.git
%cd cognivoice-component-d
!pip install -q pandas

In [ ]:
# 2. download and extract MELD into the big scratch disk (not /kaggle/working)
%cd /kaggle/tmp 2>/dev/null || %cd /tmp
!wget -q --show-progress https://huggingface.co/datasets/declare-lab/MELD/resolve/main/MELD.Raw.tar.gz
!tar -xzf MELD.Raw.tar.gz && rm MELD.Raw.tar.gz
# the inner split archives also need extracting
%cd MELD.Raw
!for f in *.tar.gz; do tar -xzf "$f" && rm "$f"; done
!ls

In [ ]:
# 3. convert mp4 -> wav and build metadata (takes a while, ~13k clips)
%cd /kaggle/working/cognivoice-component-d 2>/dev/null || %cd ~/cognivoice-component-d
MELD_ROOT = '/tmp/MELD.Raw'
WAV_DIR = '/kaggle/working/meld_wav'
!python -m src.datasets.meld --meld-root {MELD_ROOT} --wav-dir {WAV_DIR} --out /kaggle/working/metadata_meld.csv --convert

In [ ]:
# 4. fix metadata paths to be relative to the future kaggle dataset root
import pandas as pd
meta = pd.read_csv('/kaggle/working/metadata_meld.csv')
meta['path'] = meta['path'].str.replace('/kaggle/working/meld_wav', 'meld_wav', regex=False)
meta.to_csv('/kaggle/working/metadata_meld.csv', index=False)
print(len(meta), 'clips ready')
print(meta.groupby(['split', 'emotion']).size())
!du -sh /kaggle/working/meld_wav

Done. Now: Save Version (top right) -> wait for it to finish ->
open the version -> Output tab -> 'New Dataset' -> name: meld-wavs (private).